# OhioT1DM Dataset Exploration

This notebook explores the complete OhioT1DM dataset with all 24 fields including:
- **Glucose readings (CGM)** - every 5 minutes
- **Meal data** - carbs, meal type, actual meal timestamps
- **Insulin data** - basal rate, bolus dose with timestamps
- **Exercise data** - duration, intensity with timestamps
- **Physiological sensors** - heart rate, GSR, skin temp, air temp, steps
- **Event markers** - hypoglycemic events, stress events with timestamps

**New in this version:**
- Event timestamps preserved (meal_timestamp, bolus_timestamp, etc.)
- Events only appear on ONE row (cleanly mapped to nearest glucose reading)
- No forward-filling - precise event timing available

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 6)

%matplotlib inline

## 1. Load the Data

In [ ]:
# Load training data with all timestamp columns
timestamp_cols = ['timestamp', 'meal_timestamp', 'finger_stick_timestamp', 'bolus_timestamp', 
                  'exercise_timestamp', 'hypo_timestamp', 'stress_timestamp']

train_df = pd.read_csv('../data/processed/train_data.csv', parse_dates=timestamp_cols)
test_df = pd.read_csv('../data/processed/test_data.csv', parse_dates=timestamp_cols)

print(f"Training data: {train_df.shape}")
print(f"Testing data: {test_df.shape}")
print(f"\nTotal columns: {len(train_df.columns)}")
print(f"\nAll columns:")
for i, col in enumerate(train_df.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Display first few rows
train_df.head()

In [ ]:
# Data info
train_df.info()

## 2. Basic Statistics

In [ ]:
# Statistical summary
train_df.describe()

In [ ]:
# Check for missing values
missing = train_df.isnull().sum()
missing[missing > 0]

## 3. Glucose Analysis

In [ ]:
# Glucose distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(train_df['glucose'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(train_df['glucose'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["glucose"].mean():.1f} mg/dL')
axes[0].axvline(180, color='orange', linestyle='--', label='Spike threshold: 180 mg/dL')
axes[0].set_xlabel('Glucose (mg/dL)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Glucose Distribution')
axes[0].legend()

# Box plot by patient
train_df.boxplot(column='glucose', by='patient_id', ax=axes[1])
axes[1].set_xlabel('Patient ID')
axes[1].set_ylabel('Glucose (mg/dL)')
axes[1].set_title('Glucose Distribution by Patient')

plt.tight_layout()
plt.show()

In [ ]:
# Glucose over time for a single patient
patient_id = '588'
patient_data = train_df[train_df['patient_id'] == patient_id].head(500)

plt.figure(figsize=(15, 6))
plt.plot(patient_data['timestamp'], patient_data['glucose'], linewidth=1.5)
plt.axhline(180, color='red', linestyle='--', alpha=0.5, label='Spike threshold')
plt.axhline(70, color='orange', linestyle='--', alpha=0.5, label='Low threshold')
plt.xlabel('Time')
plt.ylabel('Glucose (mg/dL)')
plt.title(f'Glucose Levels Over Time - Patient {patient_id}')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Meal and Carb Analysis

In [ ]:
# Meal analysis - now with clean event markers
# Only rows with carbs > 0 are actual meal events (not forward-filled!)
meal_events = train_df[train_df['carbs'] > 0].copy()

# Calculate time difference between glucose reading and actual meal time
meal_events['time_diff_minutes'] = (
    (meal_events['timestamp'] - meal_events['meal_timestamp']).dt.total_seconds() / 60
).abs()

print(f"Total meal events: {len(meal_events)}")
print(f"Average time difference between meal and glucose reading: {meal_events['time_diff_minutes'].mean():.2f} minutes")
print(f"Max time difference: {meal_events['time_diff_minutes'].max():.2f} minutes")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Meal types
meal_counts = meal_events['meal_type'].value_counts()
meal_counts.plot(kind='bar', ax=axes[0], edgecolor='black', color='skyblue')
axes[0].set_title('Meal Types Distribution')
axes[0].set_xlabel('Meal Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Carbs distribution
axes[1].hist(meal_events['carbs'], bins=30, edgecolor='black', alpha=0.7, color='salmon')
axes[1].axvline(meal_events['carbs'].mean(), color='red', linestyle='--', 
                label=f'Mean: {meal_events["carbs"].mean():.1f}g')
axes[1].set_xlabel('Carbohydrates (g)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Carbohydrate Distribution')
axes[1].legend()

# Time difference between meal and glucose reading
axes[2].hist(meal_events['time_diff_minutes'], bins=20, edgecolor='black', alpha=0.7, color='lightgreen')
axes[2].set_xlabel('Time Difference (minutes)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Meal-to-Glucose Reading Time Difference')

plt.tight_layout()
plt.show()

print(f"\nMeal Statistics:")
print(f"  Average carbs per meal: {meal_events['carbs'].mean():.1f}g")
print(f"  Median carbs: {meal_events['carbs'].median():.1f}g")
print(f"  Min/Max carbs: {meal_events['carbs'].min():.0f}g / {meal_events['carbs'].max():.0f}g")

## 5. Physiological Sensors Analysis

In [ ]:
# Heart rate analysis
hr_data = train_df[train_df['heart_rate'] > 0]['heart_rate']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Heart rate
axes[0, 0].hist(hr_data, bins=50, edgecolor='black', alpha=0.7, color='#FF6B6B')
axes[0, 0].axvline(hr_data.mean(), color='red', linestyle='--', label=f'Mean: {hr_data.mean():.1f} bpm')
axes[0, 0].set_xlabel('Heart Rate (bpm)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Heart Rate Distribution')
axes[0, 0].legend()

# Skin temperature
skin_temp = train_df[train_df['skin_temp_f'] > 0]['skin_temp_f']
axes[0, 1].hist(skin_temp, bins=50, edgecolor='black', alpha=0.7, color='#4ECDC4')
axes[0, 1].axvline(skin_temp.mean(), color='red', linestyle='--', label=f'Mean: {skin_temp.mean():.1f}°F')
axes[0, 1].set_xlabel('Skin Temperature (°F)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Skin Temperature Distribution')
axes[0, 1].legend()

# Air temperature
air_temp = train_df[train_df['air_temp_f'] > 0]['air_temp_f']
axes[1, 0].hist(air_temp, bins=50, edgecolor='black', alpha=0.7, color='#95E1D3')
axes[1, 0].axvline(air_temp.mean(), color='red', linestyle='--', label=f'Mean: {air_temp.mean():.1f}°F')
axes[1, 0].set_xlabel('Air Temperature (°F)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Air Temperature Distribution')
axes[1, 0].legend()

# Steps
steps = train_df[train_df['steps'] > 0]['steps']
axes[1, 1].hist(steps, bins=50, edgecolor='black', alpha=0.7, color='#F38181')
axes[1, 1].axvline(steps.mean(), color='red', linestyle='--', label=f'Mean: {steps.mean():.1f} steps')
axes[1, 1].set_xlabel('Steps (per 5 min)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Step Count Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['glucose', 'carbs', 'finger_stick_glucose', 'basal_rate', 'bolus_dose', 
                'exercise_duration', 'exercise_intensity', 'heart_rate', 'gsr', 
                'skin_temp_f', 'air_temp_f', 'steps']

# Compute correlation matrix
corr_matrix = train_df[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - All Features')
plt.tight_layout()
plt.show()

## 7. Glucose Response to Meals

In [ ]:
# Glucose response to meals - using actual meal timestamps
patient_id = '588'
patient_data = train_df[train_df['patient_id'] == patient_id].copy()

# Get actual meal events (not forward-filled)
meal_events = patient_data[patient_data['carbs'] > 0].head(5)

fig, axes = plt.subplots(len(meal_events), 1, figsize=(15, 4*len(meal_events)))
if len(meal_events) == 1:
    axes = [axes]  # Make it iterable

for idx, (_, meal_row) in enumerate(meal_events.iterrows()):
    # Use ACTUAL meal timestamp, not glucose reading time
    actual_meal_time = meal_row['meal_timestamp']
    glucose_reading_time = meal_row['timestamp']
    carbs = meal_row['carbs']
    meal_type = meal_row['meal_type']
    
    # Get data 1 hour before and 3 hours after ACTUAL meal time
    start_time = actual_meal_time - pd.Timedelta(hours=1)
    end_time = actual_meal_time + pd.Timedelta(hours=3)
    
    meal_window = patient_data[
        (patient_data['timestamp'] >= start_time) & 
        (patient_data['timestamp'] <= end_time)
    ]
    
    if len(meal_window) > 0:
        ax = axes[idx]
        
        # Plot glucose
        ax.plot(meal_window['timestamp'], meal_window['glucose'], 'b-', linewidth=2, label='Glucose')
        
        # Mark actual meal time
        ax.axvline(actual_meal_time, color='red', linestyle='--', linewidth=2, 
                   alpha=0.7, label=f'Actual meal time')
        
        # Mark glucose reading that captured this meal
        ax.axvline(glucose_reading_time, color='orange', linestyle=':', linewidth=1.5,
                   alpha=0.7, label='Glucose reading (meal mapped here)')
        
        ax.axhline(180, color='purple', linestyle='--', alpha=0.5, label='Spike threshold')
        
        time_diff = abs((glucose_reading_time - actual_meal_time).total_seconds() / 60)
        ax.set_title(f'{meal_type} - {carbs}g carbs | Actual: {actual_meal_time.strftime("%H:%M")} | '
                     f'Mapped to reading: {glucose_reading_time.strftime("%H:%M")} (Δ{time_diff:.1f} min)')
        ax.set_xlabel('Time')
        ax.set_ylabel('Glucose (mg/dL)')
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ This shows how meals are mapped to the NEAREST glucose reading")
print("✓ The actual meal timestamps are preserved for precise feature engineering")

## 8. Spike Analysis

In [ ]:
# Calculate spike events (glucose > 180 mg/dL)
spike_threshold = 180
train_df['is_spike'] = (train_df['glucose'] > spike_threshold).astype(int)

# Count spikes per patient
spikes_per_patient = train_df.groupby('patient_id')['is_spike'].agg(['sum', 'mean'])
spikes_per_patient.columns = ['Total Spikes', 'Spike Rate']
spikes_per_patient['Spike Rate'] = spikes_per_patient['Spike Rate'] * 100

print("\nSpike Statistics by Patient:")
print(spikes_per_patient)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

spikes_per_patient['Total Spikes'].plot(kind='bar', ax=axes[0], edgecolor='black', color='#FF6B6B')
axes[0].set_title('Total Spike Events by Patient')
axes[0].set_xlabel('Patient ID')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

spikes_per_patient['Spike Rate'].plot(kind='bar', ax=axes[1], edgecolor='black', color='#4ECDC4')
axes[1].set_title('Spike Rate by Patient (%)')
axes[1].set_xlabel('Patient ID')
axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nOverall spike rate: {train_df['is_spike'].mean() * 100:.1f}%")

## 9. Multi-variate Analysis: Glucose vs Physiological Sensors

In [ ]:
# Sample data for plotting (to avoid overcrowding)
sample_data = train_df.sample(n=min(5000, len(train_df)), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Glucose vs Heart Rate
axes[0, 0].scatter(sample_data['heart_rate'], sample_data['glucose'], alpha=0.3, s=10)
axes[0, 0].set_xlabel('Heart Rate (bpm)')
axes[0, 0].set_ylabel('Glucose (mg/dL)')
axes[0, 0].set_title('Glucose vs Heart Rate')
axes[0, 0].grid(True, alpha=0.3)

# Glucose vs Skin Temperature
axes[0, 1].scatter(sample_data['skin_temp_f'], sample_data['glucose'], alpha=0.3, s=10, color='orange')
axes[0, 1].set_xlabel('Skin Temperature (°F)')
axes[0, 1].set_ylabel('Glucose (mg/dL)')
axes[0, 1].set_title('Glucose vs Skin Temperature')
axes[0, 1].grid(True, alpha=0.3)

# Glucose vs Steps
axes[1, 0].scatter(sample_data['steps'], sample_data['glucose'], alpha=0.3, s=10, color='green')
axes[1, 0].set_xlabel('Steps (per 5 min)')
axes[1, 0].set_ylabel('Glucose (mg/dL)')
axes[1, 0].set_title('Glucose vs Steps')
axes[1, 0].grid(True, alpha=0.3)

# Glucose vs Carbs
axes[1, 1].scatter(sample_data['carbs'], sample_data['glucose'], alpha=0.3, s=10, color='red')
axes[1, 1].set_xlabel('Carbohydrates (g)')
axes[1, 1].set_ylabel('Glucose (mg/dL)')
axes[1, 1].set_title('Glucose vs Carbs')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary Statistics

In [ ]:
print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"\nTotal Records: {len(train_df):,}")
print(f"Number of Patients: {train_df['patient_id'].nunique()}")
print(f"Date Range: {train_df['timestamp'].min()} to {train_df['timestamp'].max()}")
print(f"Duration: {(train_df['timestamp'].max() - train_df['timestamp'].min()).days} days")

print("\nGlucose Statistics:")
print(f"  Mean: {train_df['glucose'].mean():.1f} mg/dL")
print(f"  Std: {train_df['glucose'].std():.1f} mg/dL")
print(f"  Min: {train_df['glucose'].min():.1f} mg/dL")
print(f"  Max: {train_df['glucose'].max():.1f} mg/dL")
print(f"  Spike rate (>180): {(train_df['glucose'] > 180).mean() * 100:.1f}%")
print(f"  Hypo rate (<70): {(train_df['glucose'] < 70).mean() * 100:.1f}%")

print("\nMeal Statistics (CLEAN EVENT MARKERS):")
meal_events = train_df[train_df['carbs'] > 0]
print(f"  Total meal events: {len(meal_events):,} (no forward-fill duplication!)")
print(f"  Average carbs: {meal_events['carbs'].mean():.1f}g")
print(f"  Meal types: {meal_events['meal_type'].value_counts().to_dict()}")
time_diffs = (meal_events['timestamp'] - meal_events['meal_timestamp']).dt.total_seconds() / 60
print(f"  Avg time diff (meal → glucose reading): {time_diffs.abs().mean():.2f} minutes")

print("\nInsulin Statistics:")
bolus_events = train_df[train_df['bolus_dose'] > 0]
print(f"  Bolus events: {len(bolus_events):,}")
print(f"  Average bolus dose: {bolus_events['bolus_dose'].mean():.1f} units")

print("\nExercise Statistics:")
exercise_events = train_df[train_df['exercise_duration'] > 0]
print(f"  Exercise sessions: {len(exercise_events)}")
if len(exercise_events) > 0:
    print(f"  Average duration: {exercise_events['exercise_duration'].mean():.1f} minutes")
    print(f"  Average intensity: {exercise_events['exercise_intensity'].mean():.1f}/10")

print("\nPhysiological Sensors:")
print(f"  Heart rate readings: {(train_df['heart_rate'] > 0).sum():,}")
print(f"  Average heart rate: {train_df[train_df['heart_rate'] > 0]['heart_rate'].mean():.1f} bpm")
print(f"  Step count readings: {(train_df['steps'] > 0).sum():,}")
print(f"  Average steps (when active): {train_df[train_df['steps'] > 0]['steps'].mean():.1f}")

print("\nEvents:")
print(f"  Hypoglycemic events: {train_df['hypo_event'].sum():.0f}")
print(f"  Stress events: {train_df['stress_event'].sum():.0f}")

print("\n" + "=" * 70)
print("DATA STRUCTURE IMPROVEMENTS")
print("=" * 70)
print(f"✓ Events mapped to NEAREST glucose reading (not forward-filled)")
print(f"✓ Original event timestamps preserved in 6 new columns")
print(f"✓ Clean event markers: carbs > 0 only on actual meal rows")
print(f"✓ Total columns: {len(train_df.columns)} (was 18, now 24)")